# NyayaMitra — Google Colab Runner
**Indian Criminal Law RAG System**

Run cells top to bottom. Sections:
1. Setup (clone repo + install deps)
2. Mount Drive (indexes live here)
3. Test retrieval
4. Single query test
5. Batch hallucination eval
6. Full grounding/hallucination eval (paper metric)
7. LJP accuracy eval
8. Generate paper figures
9. Ingest BNSS 2023
10. Serve INLegalLlama via ngrok

---
## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Pranshu-Singh04/NyayaMitra-Proj.git"
REPO_DIR = "/content/NyayaMitra"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print("Cloned fresh repo")
else:
    !cd {REPO_DIR} && git pull
    print("Pulled latest changes")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install all dependencies
!pip install -q -r requirements.txt

# PyTorch with CUDA (Colab T4 uses cu121)
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

print("All dependencies installed")

In [ ]:
# Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 2. Mount Google Drive (Indexes Live Here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set the path to your indexes folder on Google Drive
# Change this to wherever your indexes are stored
INDEX_DIR = "/content/drive/MyDrive/NyayaMitra/indexes"

# Verify indexes exist
import os
expected = [
    "faiss_cases.index",
    "faiss_cases_metadata.jsonl",
    "faiss_statutes.index",
    "faiss_statutes_metadata.jsonl",
    "index_config.json"
]
for f in expected:
    path = os.path.join(INDEX_DIR, f)
    status = "OK" if os.path.exists(path) else "MISSING"
    size = f"({os.path.getsize(path)/1e6:.0f} MB)" if os.path.exists(path) else ""
    print(f"  [{status}] {f} {size}")

---
## 3. API Keys — Set Once, Used Everywhere

In [ ]:
# Fill in your keys here
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"   # aistudio.google.com (free)
GROQ_API_KEY   = "YOUR_GROQ_API_KEY"     # console.groq.com   (free)
GPT_API_KEY    = "YOUR_OPENAI_API_KEY"   # platform.openai.com (paid)

# Choose your default model for this session
MODEL = "gemini"   # options: gemini | groq | gpt | inlegalllama
API_KEY = GEMINI_API_KEY

print(f"Model set to: {MODEL}")

---
## 4. Test Retrieval Quality

In [ ]:
!python -u scripts/05_test_retrieval.py \
    --index_dir {INDEX_DIR} \
    --query "anticipatory bail murder Section 438 CrPC"

In [ ]:
# Try your own query
QUERY = "What is the punishment for dacoity under BNSS 2023?"

!python -u scripts/05_test_retrieval.py \
    --index_dir {INDEX_DIR} \
    --query "{QUERY}"

---
## 5. Single Query — Full Pipeline Test

In [ ]:
QUERY = "Can a person accused of murder get anticipatory bail?"

!python -u scripts/10_test_pipeline.py \
    --index_dir {INDEX_DIR} \
    --model {MODEL} \
    --api_key {API_KEY} \
    --query "{QUERY}"

In [ ]:
# Single query with hallucination check output
QUERY = "What happens if FIR is not filed within 24 hours of arrest?"

!python -u scripts/12_rag_with_hallucination.py \
    --index_dir {INDEX_DIR} \
    --model {MODEL} \
    --api_key {API_KEY} \
    --query "{QUERY}"

---
## 6. Batch Hallucination Eval

In [ ]:
# Run 10 queries — quick check
N_QUERIES = 10

!python -u scripts/12_batch_hallucination_eval.py \
    --n {N_QUERIES} \
    --model {MODEL} \
    --api_key {API_KEY} \
    --index_dir {INDEX_DIR} \
    --output_dir results/

In [ ]:
# Larger batch — 50 queries (use for paper numbers)
N_QUERIES = 50

!python -u scripts/12_batch_hallucination_eval.py \
    --n {N_QUERIES} \
    --model {MODEL} \
    --api_key {API_KEY} \
    --index_dir {INDEX_DIR} \
    --output_dir results/

In [ ]:
# Check latest results CSV
import glob, pandas as pd

files = sorted(glob.glob("results/*.csv"))
if files:
    latest = files[-1]
    print(f"Latest results: {latest}")
    df = pd.read_csv(latest)
    print(f"Total rows: {len(df)}")
    print(df.head(10))
else:
    print("No CSV results found yet")

---
## 7. Full Grounding + Hallucination Evaluation (Paper Metric)

In [ ]:
# Standard eval — 15 queries (matches paper numbers)
N_QUERIES = 15

!python -u scripts/13_evaluate_hallucination_v2.py \
    --index_dir {INDEX_DIR} \
    --model {MODEL} \
    --api_key {API_KEY} \
    --n_queries {N_QUERIES}

In [ ]:
# Full mode — eval + ablation study + LaTeX table
N_QUERIES = 15

!python -u scripts/13_evaluate_hallucination_v2.py \
    --index_dir {INDEX_DIR} \
    --model {MODEL} \
    --api_key {API_KEY} \
    --n_queries {N_QUERIES} \
    --mode full

---
## 8. LJP Accuracy vs NyayaAnumana

In [ ]:
# Set path to NyayaAnumana dataset on Drive
DATA_DIR = "/content/drive/MyDrive/NyayaMitra/data"

# Run LJP eval — 100 cases (quick)
N_CASES = 100

!python -u scripts/15_evaluate_ljp_accuracy.py \
    --index_dir {INDEX_DIR} \
    --data_dir {DATA_DIR} \
    --model {MODEL} \
    --api_key {API_KEY} \
    --n {N_CASES}

In [ ]:
# Full LJP eval — 500 cases (paper number)
N_CASES = 500

!python -u scripts/15_evaluate_ljp_accuracy.py \
    --index_dir {INDEX_DIR} \
    --data_dir {DATA_DIR} \
    --model {MODEL} \
    --api_key {API_KEY} \
    --n {N_CASES}

---
## 9. Generate Paper Figures

In [ ]:
!python -u scripts/14_generate_graphs.py \
    --results_dir results/ \
    --output_dir results/figures/

In [ ]:
# Display generated figures inline
import glob
from IPython.display import Image, display

figs = sorted(glob.glob("results/figures/*.png"))
print(f"Found {len(figs)} figures")
for fig in figs:
    print(f"\n{fig}")
    display(Image(fig))

---
## 10. Ingest BNSS 2023 into Index

In [ ]:
# Add hardcoded critical BNSS sections (no PDF needed)
!python -u scripts/16_ingest_bnss.py \
    --hardcoded \
    --index_dir {INDEX_DIR}

In [ ]:
# Ingest from a BNSS PDF (if you have it on Drive)
BNSS_PDF = "/content/drive/MyDrive/NyayaMitra/bnss2023.pdf"

!python -u scripts/16_ingest_bnss.py \
    --pdf {BNSS_PDF} \
    --index_dir {INDEX_DIR}

---
## 11. Serve INLegalLlama via ngrok
Run this section to host the model — then use `COLAB_URL` in other scripts

In [ ]:
# Install serving dependencies
!pip install -q flask pyngrok transformers accelerate bitsandbytes

In [ ]:
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"  # app.ngrok.com (free)

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
# Load INLegalLlama
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "law-ai/INLegalLlama"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("Model loaded")

In [ ]:
# Start Flask server + ngrok tunnel
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading

app = Flask(__name__)

@app.route("/generate", methods=["POST"])
def generate():
    data = request.json
    prompt = data.get("prompt", "")
    max_new_tokens = data.get("max_new_tokens", 512)
    temperature = data.get("temperature", 0.1)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return jsonify({"generated_text": generated})

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": MODEL_NAME})

# Start server in background thread
thread = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False))
thread.daemon = True
thread.start()

# Open ngrok tunnel
tunnel = ngrok.connect(5000)
COLAB_URL = tunnel.public_url
print(f"\nINLegalLlama endpoint ready!")
print(f"COLAB_URL = {COLAB_URL}")
print(f"\nUse this URL with --colab_url flag in all scripts")

In [ ]:
# Test the endpoint
import requests

resp = requests.post(f"{COLAB_URL}/generate", json={
    "prompt": "What is anticipatory bail under Indian law?",
    "max_new_tokens": 200
})
print(resp.json())

In [ ]:
# Run pipeline with INLegalLlama (set COLAB_URL from cell above)
QUERY = "Can a person accused of murder get anticipatory bail?"

!python -u scripts/10_test_pipeline.py \
    --index_dir {INDEX_DIR} \
    --model inlegalllama \
    --colab_url {COLAB_URL} \
    --query "{QUERY}"

In [ ]:
# Batch eval with INLegalLlama
N_QUERIES = 10

!python -u scripts/12_batch_hallucination_eval.py \
    --n {N_QUERIES} \
    --model inlegalllama \
    --colab_url {COLAB_URL} \
    --index_dir {INDEX_DIR} \
    --output_dir results/

---
## 12. Save Results to Google Drive

In [ ]:
import shutil, os

DRIVE_RESULTS = "/content/drive/MyDrive/NyayaMitra/results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Copy all results to Drive
!cp -r results/ /content/drive/MyDrive/NyayaMitra/
print(f"Results saved to {DRIVE_RESULTS}")

# List what was saved
import glob
saved = glob.glob(f"{DRIVE_RESULTS}/**", recursive=True)
for f in saved:
    print(f"  {f}")